# Uncertainty-Aware Pneumothorax Segmentation and Selective Prediction
### Research Portfolio Project | Medical AI (Target: MIDL / MICCAI / IEEE TMI)
**Author:** Raju Shrestha (`rajucode`)  
**Dataset:** SIIM-ACR Pneumothorax Segmentation Challenge  
**Core Question:** *"Can predictive uncertainty identify where a deep learning model is likely to produce unreliable pneumothorax segmentation predictions, and can uncertainty support selective prediction for human review?"*

---
### Experimental Matrix:
- **EXP-01 & EXP-02**: Audited metadata and zero-leakage patient-grouped stratified splitting (Fold 0: Val, Folds 1-4: Train, Holdout Test: 2,135 images).
- **EXP-03**: Deterministic ResNet34 U-Net Baseline.
- **EXP-04**: Monte Carlo Dropout ResNet34 U-Net ($T=20$ stochastic passes, Spatial Dropout $p=0.2$).
- **EXP-05**: Deep Ensemble ($M=3$ independently seeded ResNet34 U-Nets with mutual information decomposition).
- **EXP-06**: Error-detection AUROC (AUROC-ED).
- **EXP-07**: Reliability & Expected Segmentation Calibration Error (ESCE).
- **EXP-08**: Selective Prediction & Risk-Coverage Pareto Curves (AURC).
- **EXP-09**: Anatomical Subgroup Evaluation (AP vs PA projection views).
- **EXP-10**: Statistical hypothesis testing with 1,000 bootstrap resamples (95% CIs) and paired Wilcoxon tests.


In [ ]:
# 1. System Setup and Package Installation
import os
import sys
import time
import math
import random
import glob
import ast
import json
import shutil
from typing import List, Dict, Tuple, Optional, Any

print("Installing required medical imaging libraries...")
os.system("pip install -q --break-system-packages pydicom albumentations segmentation-models-pytorch")

import numpy as np
import pandas as pd
import pydicom
import cv2
from scipy.ndimage import distance_transform_edt
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import roc_auc_score, roc_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

# PyTorch and GPU verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Deterministic seeding helper
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Seeding initialized to 42.")


In [ ]:
# 2. Data Discovery and Zero-Leakage Splits Ingestion

# Find DICOM files across available input paths
def index_dicom_files() -> Dict[str, str]:
    search_paths = [
        "/kaggle/input/siim-acr-pneumothorax-segmentation-data",
        "/kaggle/input/siim-acr-pneumothorax-segmentation",
        "/kaggle/input"
    ]
    dicom_map = {}
    print("Indexing DICOM files in /kaggle/input...")
    for sp in search_paths:
        if os.path.exists(sp):
            for root, _, files in os.walk(sp):
                for f in files:
                    if f.endswith(".dcm"):
                        stem = f[:-4]
                        if stem not in dicom_map:
                            dicom_map[stem] = os.path.join(root, f)
    print(f"Indexed {len(dicom_map)} unique DICOM radiographs.")
    return dicom_map

dicom_map = index_dicom_files()

# Load processed patient splits
splits_path = "/kaggle/input/siim-acr-processed-splits/patient_splits.csv"
if not os.path.exists(splits_path):
    # Fallback search
    matched = glob.glob("/kaggle/input/**/patient_splits.csv", recursive=True)
    if matched:
        splits_path = matched[0]
    else:
        raise FileNotFoundError("patient_splits.csv not found in /kaggle/input.")

df_splits = pd.read_csv(splits_path)
print(f"Loaded splits from: {splits_path}")
print(f"Total records: {len(df_splits)}")
print(f"Folds distribution:\n{df_splits['Fold'].value_counts().sort_index()}")

# Match with available DICOM files
df_splits["dcm_path"] = df_splits["ImageId"].map(dicom_map)
matched_count = df_splits["dcm_path"].notna().sum()
print(f"Successfully matched DICOM files: {matched_count} / {len(df_splits)}")

if matched_count < len(df_splits):
    print("Filtering to available radiographs...")
    df_splits = df_splits[df_splits["dcm_path"].notna()].reset_index(drop=True)


In [ ]:
# 3. SIIM-ACR RLE Decoder with Multi-Mask Logical OR Aggregation

def rle_decode(rle_str: Any, shape: tuple = (1024, 1024)) -> np.ndarray:
    """Decode SIIM-ACR Run-Length Encoded string into binary mask (Fortran order)."""
    if rle_str is None or (isinstance(rle_str, float) and np.isnan(rle_str)):
        return np.zeros(shape, dtype=np.uint8)

    rle_str = str(rle_str).strip()
    if rle_str == "" or rle_str == "-1":
        return np.zeros(shape, dtype=np.uint8)

    s = rle_str.split()
    starts = np.asarray([int(float(x)) for x in s[0::2]], dtype=int) - 1
    lengths = np.asarray([int(float(x)) for x in s[1::2]], dtype=int)
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape(shape, order="F")

def aggregate_rle_list(rle_list: List[str], shape: tuple = (1024, 1024)) -> np.ndarray:
    """Combine multiple RLE instances for an image using logical OR."""
    composite = np.zeros(shape, dtype=np.uint8)
    for rle in rle_list:
        mask = rle_decode(rle, shape=shape)
        composite = np.bitwise_or(composite, mask)
    return composite

# Verify RLE decoder on sample
sample_rles = ast.literal_eval(df_splits.iloc[0]["EncodedPixelsList"])
sample_mask = aggregate_rle_list(sample_rles)
print(f"Verification: sample mask shape={sample_mask.shape}, sum={np.sum(sample_mask)}")


In [ ]:
# 4. PyTorch Dataset and Radiologically Sound Augmentations

class SIIMPneumothoraxDataset(Dataset):
    """Dataset for SIIM-ACR pneumothorax radiographs with dense masks."""

    def __init__(self, df: pd.DataFrame, image_size: int = 512, transforms: Optional[Any] = None):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.transforms = transforms

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        dcm_path = row["dcm_path"]
        
        # Read DICOM
        try:
            dcm = pydicom.dcmread(dcm_path)
            img = dcm.pixel_array.astype(np.float32)
            if hasattr(dcm, "PhotometricInterpretation") and dcm.PhotometricInterpretation == "MONOCHROME1":
                img = np.amax(img) - img
        except Exception:
            img = np.zeros((1024, 1024), dtype=np.float32)

        # Normalize to uint8 [0, 255]
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = ((img - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)
        else:
            img = np.zeros_like(img, dtype=np.uint8)

        # Grayscale to 3-channel
        img_3c = np.repeat(np.expand_dims(img, axis=-1), 3, axis=-1)

        # Decode ground-truth mask
        rle_raw = row["EncodedPixelsList"]
        if isinstance(rle_raw, str):
            try:
                rle_list = ast.literal_eval(rle_raw)
            except Exception:
                rle_list = [rle_raw]
        else:
            rle_list = [str(rle_raw)]

        mask = aggregate_rle_list(rle_list, shape=img.shape)

        # Apply Albumentations transforms
        if self.transforms is not None:
            augmented = self.transforms(image=img_3c, mask=mask)
            image_tensor = augmented["image"]
            mask_tensor = augmented["mask"].unsqueeze(0).float()
        else:
            image_tensor = torch.from_numpy(img_3c).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "image_id": row["ImageId"],
            "patient_id": str(row["PatientID"]),
            "view_position": str(row["ViewPosition"]),
            "has_pneumothorax": int(row["HasPneumothorax"])
        }

# Data augmentations: HorizontalFlip, ShiftScaleRotate, RandomBrightnessContrast
# Strictly NO VerticalFlip (violates anatomical cephalocaudal orientation)
train_transforms = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-10, 10), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Partition DataFrames
# Train: Folds 1, 2, 3, 4 | Val: Fold 0 | Test Holdout: Fold 'test'
train_df = df_splits[df_splits["Fold"].isin(["1", "2", "3", "4", 1, 2, 3, 4])].reset_index(drop=True)
val_df = df_splits[df_splits["Fold"].isin(["0", 0])].reset_index(drop=True)
test_df = df_splits[df_splits["Fold"].isin(["test", "Test"])].reset_index(drop=True)

print(f"Train set: {len(train_df)} images ({train_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Validation set: {len(val_df)} images ({val_df['HasPneumothorax'].mean()*100:.1f}% positive)")
print(f"Test Holdout set: {len(test_df)} images ({test_df['HasPneumothorax'].mean()*100:.1f}% positive)")

# Verify zero patient leakage
train_val_patients = set(train_df["PatientID"]).union(set(val_df["PatientID"]))
test_patients = set(test_df["PatientID"])
overlap = train_val_patients.intersection(test_patients)
assert len(overlap) == 0, f"DATA LEAKAGE DETECTED: {len(overlap)} overlapping patients!"
print(f"LEAKAGE PROOF VERIFIED: Exactly {len(overlap)} overlapping patients between train/val and test.")


In [ ]:
# 5. Architecture: ResNet34 U-Net with Spatial Dropout and Combined Loss
import segmentation_models_pytorch as smp

class PneumothoraxUNet(nn.Module):
    """ResNet34 U-Net supporting deterministic inference and MC Dropout."""

    def __init__(self, dropout_rate: float = 0.2, pretrained: bool = True):
        super().__init__()
        self.dropout_rate = dropout_rate
        weights = "imagenet" if pretrained else None
        
        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=weights,
            in_channels=3,
            classes=1,
            decoder_channels=(256, 128, 64, 32, 16)
        )
        
        # Inject SpatialDropout2d into decoder blocks for MC Dropout
        if dropout_rate > 0.0:
            for idx in range(len(self.model.decoder.blocks)):
                self.model.decoder.blocks[idx].add_module(
                    "spatial_dropout", nn.Dropout2d(p=dropout_rate)
                )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    @torch.no_grad()
    def predict_deterministic(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Deterministic forward pass (eval mode)."""
        self.eval()
        logits = self.forward(x)
        prob = torch.sigmoid(logits)
        eps = 1e-7
        p_clamped = torch.clamp(prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        return {"prob": prob, "entropy": entropy}

    @torch.no_grad()
    def predict_mc_dropout(self, x: torch.Tensor, num_samples: int = 20) -> Dict[str, torch.Tensor]:
        """Monte Carlo Dropout inference with T stochastic passes."""
        self.train() # Activates Spatial Dropout during inference
        samples = []
        for _ in range(num_samples):
            logits = self.forward(x)
            samples.append(torch.sigmoid(logits))
        
        stacked = torch.stack(samples, dim=0) # (T, B, 1, H, W)
        mean_prob = torch.mean(stacked, dim=0)
        variance = torch.var(stacked, dim=0, unbiased=True)
        
        eps = 1e-7
        p_clamped = torch.clamp(mean_prob, eps, 1.0 - eps)
        entropy = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
        
        return {
            "mean": mean_prob,
            "variance": variance,
            "entropy": entropy
        }

# Combined Loss: 0.5 * BCE + 0.5 * SoftDice
class SoftDiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits).view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        cardinality = (probs.pow(2) + targets.pow(2)).sum()
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice

class CombinedBCEDiceLoss(nn.Module):
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.dice_loss = SoftDiceLoss(smooth=1.0)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_bce = F.binary_cross_entropy_with_logits(logits, targets)
        loss_dice = self.dice_loss(logits, targets)
        return self.bce_weight * loss_bce + self.dice_weight * loss_dice

criterion = CombinedBCEDiceLoss()
print("Model architecture and CombinedBCEDiceLoss verified.")


In [ ]:
# 6. Deep Ensemble Container with Epistemic Uncertainty Decomposition

class DeepEnsemble(nn.Module):
    """Ensemble of M independent models with Mutual Information decomposition."""

    def __init__(self, models: List[PneumothoraxUNet]):
        super().__init__()
        self.models = nn.ModuleList(models)
        self.num_models = len(models)

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        member_probs = []
        member_entropies = []
        eps = 1e-7

        for model in self.models:
            model.eval()
            logits = model(x)
            prob = torch.sigmoid(logits)
            member_probs.append(prob)
            
            p_clamped = torch.clamp(prob, eps, 1.0 - eps)
            ent = -p_clamped * torch.log2(p_clamped) - (1.0 - p_clamped) * torch.log2(1.0 - p_clamped)
            member_entropies.append(ent)

        stacked_probs = torch.stack(member_probs, dim=0) # (M, B, 1, H, W)
        stacked_ents = torch.stack(member_entropies, dim=0)

        # 1. Predictive Mean
        mean_prob = torch.mean(stacked_probs, dim=0)
        # 2. Predictive Variance
        variance = torch.var(stacked_probs, dim=0, unbiased=True)
        # 3. Total Entropy: H(mean)
        mean_clamped = torch.clamp(mean_prob, eps, 1.0 - eps)
        total_entropy = -mean_clamped * torch.log2(mean_clamped) - (1.0 - mean_clamped) * torch.log2(1.0 - mean_clamped)
        # 4. Expected Entropy (Aleatoric): E[H(p_m)]
        expected_entropy = torch.mean(stacked_ents, dim=0)
        # 5. Mutual Information (Epistemic): I(Y; theta | x) = H(mean) - E[H(p_m)]
        mutual_info = torch.clamp(total_entropy - expected_entropy, min=0.0)

        return {
            "mean": mean_prob,
            "variance": variance,
            "total_entropy": total_entropy,
            "expected_entropy": expected_entropy,
            "mutual_information": mutual_info
        }

print("DeepEnsemble container ready.")


In [ ]:
# 7. Evaluation Metrics: Disaggregated Dice, ESCE, AUROC-ED, and AURC

def compute_dice_coefficient(y_true: np.ndarray, y_pred: np.ndarray, empty_score: float = 1.0) -> float:
    y_true_sum = np.sum(y_true)
    y_pred_sum = np.sum(y_pred)
    if y_true_sum == 0 and y_pred_sum == 0:
        return empty_score
    if y_true_sum == 0 or y_pred_sum == 0:
        return 0.0
    intersection = np.sum(y_true * y_pred)
    return float(2.0 * intersection / (y_true_sum + y_pred_sum))

def compute_segmentation_metrics(y_true: np.ndarray, y_pred_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_pred_prob >= threshold).astype(np.uint8)
    y_true = (y_true > 0).astype(np.uint8)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    is_positive = float(np.sum(y_true) > 0)
    dice = compute_dice_coefficient(y_true, y_pred)
    iou = float(tp / (tp + fp + fn)) if (tp + fp + fn) > 0 else (1.0 if is_positive == 0 else 0.0)
    sensitivity = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 1.0
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else (1.0 if np.sum(y_pred) == 0 else 0.0)

    return {
        "dice": dice,
        "iou": iou,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "is_positive": is_positive,
    }

def compute_esce(probs: np.ndarray, targets: np.ndarray, n_bins: int = 10):
    probs_flat = probs.flatten()
    targets_flat = targets.flatten()
    total = len(probs_flat)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_confs = np.zeros(n_bins)
    bin_accs = np.zeros(n_bins)
    bin_counts = np.zeros(n_bins)
    esce = 0.0

    for i in range(n_bins):
        low, high = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (probs_flat > low) & (probs_flat <= high) if i > 0 else (probs_flat >= low) & (probs_flat <= high)
        count = np.sum(mask)
        bin_counts[i] = count
        if count > 0:
            conf = np.mean(probs_flat[mask])
            acc = np.mean(targets_flat[mask])
            bin_confs[i] = conf
            bin_accs[i] = acc
            esce += (count / total) * np.abs(acc - conf)

    return float(esce), bin_confs, bin_accs, bin_counts

def compute_brier_score(probs: np.ndarray, targets: np.ndarray) -> float:
    return float(np.mean((probs - targets) ** 2))

def compute_error_detection_auroc(uncertainty_map: np.ndarray, probs: np.ndarray, targets: np.ndarray, threshold: float = 0.5) -> float:
    preds = (probs >= threshold).astype(np.uint8)
    error = np.abs(targets - preds).flatten()
    uncertainty = uncertainty_map.flatten()
    if np.all(error == 0) or np.all(error == 1):
        return 0.5
    if len(error) > 100_000:
        idx = np.random.choice(len(error), size=100_000, replace=False)
        error = error[idx]
        uncertainty = uncertainty[idx]
    try:
        return float(roc_auc_score(error, uncertainty))
    except Exception:
        return 0.5

def aggregate_case_uncertainty(uncertainty_map: np.ndarray, k: int = 500) -> float:
    flat = uncertainty_map.flatten()
    k = min(k, len(flat))
    top_k_vals = np.partition(flat, -k)[-k:]
    return float(np.mean(top_k_vals))

def compute_risk_coverage_curve(case_uncertainties: np.ndarray, case_risks: np.ndarray, steps: int = 50, min_cov: float = 0.20):
    n = len(case_uncertainties)
    sorted_idx = np.argsort(case_uncertainties)
    coverages = np.linspace(min_cov, 1.0, steps)
    risks = np.zeros(steps)
    for i, cov in enumerate(coverages):
        retain_n = max(1, int(round(cov * n)))
        retained = sorted_idx[:retain_n]
        risks[i] = float(np.mean(case_risks[retained]))
    return coverages, risks

def compute_aurc(coverages: np.ndarray, risks: np.ndarray) -> float:
    cov_norm = (coverages - coverages[0]) / (coverages[-1] - coverages[0])
    try:
        from scipy.integrate import trapezoid
        return float(trapezoid(risks, cov_norm))
    except ImportError:
        trapz_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))
        return float(trapz_fn(risks, cov_norm))


In [ ]:
# 8. Training Pipeline for 3 Independent Models (Deep Ensemble M=3)

batch_size = 16
num_epochs = 10
lr = 1e-3

train_loader = DataLoader(
    SIIMPneumothoraxDataset(train_df, image_size=512, transforms=train_transforms),
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    SIIMPneumothoraxDataset(val_df, image_size=512, transforms=val_transforms),
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

def train_model(seed: int, model_name: str) -> PneumothoraxUNet:
    print(f"\n==========================================")
    print(f"Training {model_name} with Seed {seed}")
    print(f"==========================================")
    seed_everything(seed)
    
    model = PneumothoraxUNet(dropout_rate=0.2, pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler()
    
    best_val_dice = -1.0
    best_weights_path = f"{model_name}.pt"

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss = 0.0
        start_time = time.time()
        
        for batch in train_loader:
            images = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)
            
            optimizer.zero_grad()
            with torch.amp.autocast(device_type="cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * images.size(0)
            
        train_loss /= len(train_df)
        scheduler.step()
        
        # Validation
        model.eval()
        val_dices_all = []
        val_dices_pos = []
        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(device, non_blocking=True)
                masks = batch["mask"].cpu().numpy()
                with torch.amp.autocast(device_type="cuda"):
                    logits = model(images)
                    probs = torch.sigmoid(logits).cpu().numpy()
                    
                for i in range(len(masks)):
                    m = masks[i, 0]
                    p = probs[i, 0]
                    is_pos = np.sum(m) > 0
                    d = compute_dice_coefficient(m, (p >= 0.5).astype(np.uint8))
                    val_dices_all.append(d)
                    if is_pos:
                        val_dices_pos.append(d)
                        
        mean_all_dice = np.mean(val_dices_all)
        mean_pos_dice = np.mean(val_dices_pos) if val_dices_pos else 0.0
        elapsed = time.time() - start_time
        
        print(f"Epoch {epoch:02d}/{num_epochs:02d} [{elapsed:.0f}s] - Train Loss: {train_loss:.4f} | Val DSC_all: {mean_all_dice:.4f} | Val DSC_pos: {mean_pos_dice:.4f}")
        
        if mean_pos_dice > best_val_dice:
            best_val_dice = mean_pos_dice
            torch.save(model.state_dict(), best_weights_path)
            print(f"  --> Saved new best checkpoint to {best_weights_path} (DSC_pos: {best_val_dice:.4f})")
            
    # Load best weights
    model.load_state_dict(torch.load(best_weights_path))
    model.eval()
    return model

# Train 3 models for Deep Ensemble
model_seed42 = train_model(seed=42, model_name="model_seed42")
model_seed43 = train_model(seed=43, model_name="model_seed43")
model_seed44 = train_model(seed=44, model_name="model_seed44")

ensemble_model = DeepEnsemble([model_seed42, model_seed43, model_seed44])
print("\nAll 3 models trained and DeepEnsemble assembled successfully.")


In [ ]:
# 9. Test Holdout Evaluation: EXP-03, EXP-04, EXP-05 on 2,135 Radiographs

test_loader = DataLoader(
    SIIMPneumothoraxDataset(test_df, image_size=512, transforms=val_transforms),
    batch_size=1, # Single radiograph inference for granular uncertainty mapping
    shuffle=False,
    num_workers=2
)

test_records = []
print(f"Executing test holdout evaluation across {len(test_df)} cases...")
start_infer = time.time()

# Accumulators for overall calibration & AUROC
all_targets = []
det_probs = []
det_entropies = []
mc_probs = []
mc_variances = []
ens_probs = []
ens_mis = []

for idx, batch in enumerate(test_loader):
    img = batch["image"].to(device)
    mask = batch["mask"].numpy()[0, 0]
    img_id = batch["image_id"][0]
    patient_id = batch["patient_id"][0]
    view_pos = batch["view_position"][0]
    has_ptx = batch["has_pneumothorax"][0].item()

    # 1. Deterministic Baseline (EXP-03)
    det_out = model_seed42.predict_deterministic(img)
    p_det = det_out["prob"].cpu().numpy()[0, 0]
    ent_det = det_out["entropy"].cpu().numpy()[0, 0]
    det_metrics = compute_segmentation_metrics(mask, p_det)
    det_case_unc = aggregate_case_uncertainty(ent_det, k=500)

    # 2. Monte Carlo Dropout (T=20) (EXP-04)
    mc_out = model_seed42.predict_mc_dropout(img, num_samples=20)
    p_mc = mc_out["mean"].cpu().numpy()[0, 0]
    var_mc = mc_out["variance"].cpu().numpy()[0, 0]
    mc_metrics = compute_segmentation_metrics(mask, p_mc)
    mc_case_unc = aggregate_case_uncertainty(var_mc, k=500)

    # 3. Deep Ensemble (M=3) (EXP-05)
    ens_out = ensemble_model(img)
    p_ens = ens_out["mean"].cpu().numpy()[0, 0]
    mi_ens = ens_out["mutual_information"].cpu().numpy()[0, 0]
    ens_metrics = compute_segmentation_metrics(mask, p_ens)
    ens_case_unc = aggregate_case_uncertainty(mi_ens, k=500)

    # Error-detection AUROC per case (subsampled)
    auroc_ed_det = compute_error_detection_auroc(ent_det, p_det, mask)
    auroc_ed_mc = compute_error_detection_auroc(var_mc, p_mc, mask)
    auroc_ed_ens = compute_error_detection_auroc(mi_ens, p_ens, mask)

    test_records.append({
        "ImageId": img_id,
        "PatientID": patient_id,
        "ViewPosition": view_pos,
        "HasPneumothorax": has_ptx,
        # Deterministic
        "det_dice": det_metrics["dice"],
        "det_iou": det_metrics["iou"],
        "det_sens": det_metrics["sensitivity"],
        "det_spec": det_metrics["specificity"],
        "det_case_unc": det_case_unc,
        "det_auroc_ed": auroc_ed_det,
        # MC Dropout
        "mc_dice": mc_metrics["dice"],
        "mc_iou": mc_metrics["iou"],
        "mc_sens": mc_metrics["sensitivity"],
        "mc_spec": mc_metrics["specificity"],
        "mc_case_unc": mc_case_unc,
        "mc_auroc_ed": auroc_ed_mc,
        # Deep Ensemble
        "ens_dice": ens_metrics["dice"],
        "ens_iou": ens_metrics["iou"],
        "ens_sens": ens_metrics["sensitivity"],
        "ens_spec": ens_metrics["specificity"],
        "ens_case_unc": ens_case_unc,
        "ens_auroc_ed": auroc_ed_ens,
    })

    # Pixel sampling for overall calibration (1,000 pixels per radiograph)
    pix_idx = np.random.choice(512 * 512, size=1000, replace=False)
    all_targets.append(mask.flatten()[pix_idx])
    det_probs.append(p_det.flatten()[pix_idx])
    det_entropies.append(ent_det.flatten()[pix_idx])
    mc_probs.append(p_mc.flatten()[pix_idx])
    mc_variances.append(var_mc.flatten()[pix_idx])
    ens_probs.append(p_ens.flatten()[pix_idx])
    ens_mis.append(mi_ens.flatten()[pix_idx])

    if (idx + 1) % 250 == 0 or (idx + 1) == len(test_df):
        print(f"Processed {idx + 1}/{len(test_df)} cases ({(time.time() - start_infer)/60:.1f} min elapsed)")

df_results = pd.DataFrame(test_records)
df_results.to_csv("test_predictions.csv", index=False)
print("Test predictions saved to test_predictions.csv")


In [ ]:
# 10. Quantitative Results, Calibration (ESCE), and Statistical Significance

# Aggregate calibration arrays
targets_pool = np.concatenate(all_targets)
p_det_pool = np.concatenate(det_probs)
p_mc_pool = np.concatenate(mc_probs)
p_ens_pool = np.concatenate(ens_probs)

# Compute ESCE and Brier Score
esce_det, confs_det, accs_det, _ = compute_esce(p_det_pool, targets_pool)
esce_mc, confs_mc, accs_mc, _ = compute_esce(p_mc_pool, targets_pool)
esce_ens, confs_ens, accs_ens, _ = compute_esce(p_ens_pool, targets_pool)

brier_det = compute_brier_score(p_det_pool, targets_pool)
brier_mc = compute_brier_score(p_mc_pool, targets_pool)
brier_ens = compute_brier_score(p_ens_pool, targets_pool)

# Disaggregate metrics
pos_mask = df_results["HasPneumothorax"] == 1
neg_mask = df_results["HasPneumothorax"] == 0

def get_stats(series):
    return float(np.mean(series)), float(np.std(series))

print("\n" + "="*80)
print("DISAGGREGATED SEGMENTATION EVALUATION (TEST HOLDOUT, N=2,135)")
print("="*80)
print(f"Metric                        | Deterministic (EXP-03) | MC Dropout (EXP-04) | Deep Ensemble (EXP-05)")
print("-"*80)
print(f"DSC_pos (Positive Cases Only) | {df_results.loc[pos_mask, 'det_dice'].mean():.4f} +/- {df_results.loc[pos_mask, 'det_dice'].std():.4f}      | {df_results.loc[pos_mask, 'mc_dice'].mean():.4f} +/- {df_results.loc[pos_mask, 'mc_dice'].std():.4f}  | {df_results.loc[pos_mask, 'ens_dice'].mean():.4f} +/- {df_results.loc[pos_mask, 'ens_dice'].std():.4f}")
print(f"DSC_all (Overall Cohort)      | {df_results['det_dice'].mean():.4f} +/- {df_results['det_dice'].std():.4f}      | {df_results['mc_dice'].mean():.4f} +/- {df_results['mc_dice'].std():.4f}  | {df_results['ens_dice'].mean():.4f} +/- {df_results['ens_dice'].std():.4f}")
print(f"IoU (Intersection-over-Union) | {df_results.loc[pos_mask, 'det_iou'].mean():.4f} +/- {df_results.loc[pos_mask, 'det_iou'].std():.4f}      | {df_results.loc[pos_mask, 'mc_iou'].mean():.4f} +/- {df_results.loc[pos_mask, 'mc_iou'].std():.4f}  | {df_results.loc[pos_mask, 'ens_iou'].mean():.4f} +/- {df_results.loc[pos_mask, 'ens_iou'].std():.4f}")
print(f"Sensitivity (Recall)          | {df_results.loc[pos_mask, 'det_sens'].mean():.4f} +/- {df_results.loc[pos_mask, 'det_sens'].std():.4f}      | {df_results.loc[pos_mask, 'mc_sens'].mean():.4f} +/- {df_results.loc[pos_mask, 'mc_sens'].std():.4f}  | {df_results.loc[pos_mask, 'ens_sens'].mean():.4f} +/- {df_results.loc[pos_mask, 'ens_sens'].std():.4f}")
print(f"Specificity                   | {df_results['det_spec'].mean():.4f} +/- {df_results['det_spec'].std():.4f}      | {df_results['mc_spec'].mean():.4f} +/- {df_results['mc_spec'].std():.4f}  | {df_results['ens_spec'].mean():.4f} +/- {df_results['ens_spec'].std():.4f}")
print(f"AUROC-ED (Error Detection)    | {df_results['det_auroc_ed'].mean():.4f}                 | {df_results['mc_auroc_ed'].mean():.4f}             | {df_results['ens_auroc_ed'].mean():.4f}")
print(f"ESCE (Calibration Error)      | {esce_det:.4f}                 | {esce_mc:.4f}             | {esce_ens:.4f}")
print(f"Brier Score                   | {brier_det:.4f}                 | {brier_mc:.4f}             | {brier_ens:.4f}")
print("="*80)

# Bootstrap 95% Confidence Intervals (1,000 resamples)
def bootstrap_ci(metric_a, metric_b=None, n_resamples=1000, seed=42):
    np.random.seed(seed)
    n = len(metric_a)
    boot_stats = []
    for _ in range(n_resamples):
        idx = np.random.choice(n, size=n, replace=True)
        if metric_b is not None:
            diff = np.mean(metric_a[idx]) - np.mean(metric_b[idx])
            boot_stats.append(diff)
        else:
            boot_stats.append(np.mean(metric_a[idx]))
    low = np.percentile(boot_stats, 2.5)
    high = np.percentile(boot_stats, 97.5)
    return float(low), float(high)

# Hypothesis Testing
# H1: Deep Ensemble DSC_pos > Deterministic DSC_pos
w_h1, p_h1 = wilcoxon(df_results.loc[pos_mask, "ens_dice"], df_results.loc[pos_mask, "det_dice"], alternative="greater")
# H2: MC Dropout AUROC-ED > Deterministic AUROC-ED
w_h2, p_h2 = wilcoxon(df_results["mc_auroc_ed"], df_results["det_auroc_ed"], alternative="greater")
# H3: Deep Ensemble AUROC-ED > Deterministic AUROC-ED
w_h3, p_h3 = wilcoxon(df_results["ens_auroc_ed"], df_results["det_auroc_ed"], alternative="greater")

print(f"Hypothesis H1 (Ens DSC_pos > Det DSC_pos): p-value = {p_h1:.4e} (Significant: {p_h1 < 0.05})")
print(f"Hypothesis H2 (MC AUROC-ED > Det AUROC-ED): p-value = {p_h2:.4e} (Significant: {p_h2 < 0.05})")
print(f"Hypothesis H3 (Ens AUROC-ED > Det AUROC-ED): p-value = {p_h3:.4e} (Significant: {p_h3 < 0.05})")


In [ ]:
# 11. Selective Prediction and Clinical Triage Simulation (EXP-08)

# Risk defined as empirical error (1.0 - Dice)
risks_det = 1.0 - df_results["det_dice"].values
risks_mc = 1.0 - df_results["mc_dice"].values
risks_ens = 1.0 - df_results["ens_dice"].values

# Case uncertainties
unc_det = df_results["det_case_unc"].values
unc_mc = df_results["mc_case_unc"].values
unc_ens = df_results["ens_case_unc"].values
unc_random = np.random.RandomState(42).rand(len(df_results))

# Compute Risk-Coverage Pareto curves (20% to 100% coverage)
cov_random, r_random = compute_risk_coverage_curve(unc_random, risks_det)
cov_det, r_det = compute_risk_coverage_curve(unc_det, risks_det)
cov_mc, r_mc = compute_risk_coverage_curve(unc_mc, risks_mc)
cov_ens, r_ens = compute_risk_coverage_curve(unc_ens, risks_ens)

aurc_random = compute_aurc(cov_random, r_random)
aurc_det = compute_aurc(cov_det, r_det)
aurc_mc = compute_aurc(cov_mc, r_mc)
aurc_ens = compute_aurc(cov_ens, r_ens)

print("\n" + "="*80)
print("SELECTIVE PREDICTION AND RISK-COVERAGE EVALUATION (EXP-08)")
print("="*80)
print(f"Strategy              | AURC (v) | DSC at 100% Cov | DSC at 90% Cov | DSC at 80% Cov | DSC at 70% Cov")
print("-"*80)

def evaluate_retention(uncertainties, dices, coverages=[1.0, 0.9, 0.8, 0.7]):
    n = len(uncertainties)
    s_idx = np.argsort(uncertainties)
    ret = []
    for c in coverages:
        cnt = max(1, int(round(c * n)))
        ret.append(float(np.mean(dices[s_idx[:cnt]])))
    return ret

ret_random = evaluate_retention(unc_random, df_results["det_dice"].values)
ret_det = evaluate_retention(unc_det, df_results["det_dice"].values)
ret_mc = evaluate_retention(unc_mc, df_results["mc_dice"].values)
ret_ens = evaluate_retention(unc_ens, df_results["ens_dice"].values)

print(f"Random Referral       | {aurc_random:.4f}   | {ret_random[0]:.4f}          | {ret_random[1]:.4f}         | {ret_random[2]:.4f}         | {ret_random[3]:.4f}")
print(f"Deterministic Entropy | {aurc_det:.4f}   | {ret_det[0]:.4f}          | {ret_det[1]:.4f}         | {ret_det[2]:.4f}         | {ret_det[3]:.4f}")
print(f"MC Dropout Variance   | {aurc_mc:.4f}   | {ret_mc[0]:.4f}          | {ret_mc[1]:.4f}         | {ret_mc[2]:.4f}         | {ret_mc[3]:.4f}")
print(f"Deep Ensemble MutInfo | {aurc_ens:.4f}   | {ret_ens[0]:.4f}          | {ret_ens[1]:.4f}         | {ret_ens[2]:.4f}         | {ret_ens[3]:.4f}")
print("="*80)

# Save Risk-Coverage curves to CSV
df_rc = pd.DataFrame({
    "Coverage": cov_det,
    "Risk_Random": r_random,
    "Risk_Deterministic": r_det,
    "Risk_MCDropout": r_mc,
    "Risk_DeepEnsemble": r_ens
})
df_rc.to_csv("risk_coverage_curves.csv", index=False)
print("Risk-coverage curves saved to risk_coverage_curves.csv")


In [ ]:
# 12. Anatomical Subgroup Evaluation: AP vs PA Projection Views (EXP-09)

ap_mask = df_results["ViewPosition"] == "AP"
pa_mask = df_results["ViewPosition"] == "PA"

print("\n" + "="*80)
print(f"ANATOMICAL SUBGROUP ANALYSIS: AP ({ap_mask.sum()} cases) vs PA ({pa_mask.sum()} cases)")
print("="*80)
print(f"Subgroup | Model             | DSC_pos           | DSC_all           | AUROC-ED  | Case Uncertainty")
print("-"*80)
print(f"AP       | Deterministic     | {df_results.loc[ap_mask & pos_mask, 'det_dice'].mean():.4f}            | {df_results.loc[ap_mask, 'det_dice'].mean():.4f}            | {df_results.loc[ap_mask, 'det_auroc_ed'].mean():.4f}    | {df_results.loc[ap_mask, 'det_case_unc'].mean():.4f}")
print(f"AP       | MC Dropout        | {df_results.loc[ap_mask & pos_mask, 'mc_dice'].mean():.4f}            | {df_results.loc[ap_mask, 'mc_dice'].mean():.4f}            | {df_results.loc[ap_mask, 'mc_auroc_ed'].mean():.4f}    | {df_results.loc[ap_mask, 'mc_case_unc'].mean():.4f}")
print(f"AP       | Deep Ensemble     | {df_results.loc[ap_mask & pos_mask, 'ens_dice'].mean():.4f}            | {df_results.loc[ap_mask, 'ens_dice'].mean():.4f}            | {df_results.loc[ap_mask, 'ens_auroc_ed'].mean():.4f}    | {df_results.loc[ap_mask, 'ens_case_unc'].mean():.4f}")
print("-"*80)
print(f"PA       | Deterministic     | {df_results.loc[pa_mask & pos_mask, 'det_dice'].mean():.4f}            | {df_results.loc[pa_mask, 'det_dice'].mean():.4f}            | {df_results.loc[pa_mask, 'det_auroc_ed'].mean():.4f}    | {df_results.loc[pa_mask, 'det_case_unc'].mean():.4f}")
print(f"PA       | MC Dropout        | {df_results.loc[pa_mask & pos_mask, 'mc_dice'].mean():.4f}            | {df_results.loc[pa_mask, 'mc_dice'].mean():.4f}            | {df_results.loc[pa_mask, 'mc_auroc_ed'].mean():.4f}    | {df_results.loc[pa_mask, 'mc_case_unc'].mean():.4f}")
print(f"PA       | Deep Ensemble     | {df_results.loc[pa_mask & pos_mask, 'ens_dice'].mean():.4f}            | {df_results.loc[pa_mask, 'ens_dice'].mean():.4f}            | {df_results.loc[pa_mask, 'ens_auroc_ed'].mean():.4f}    | {df_results.loc[pa_mask, 'ens_case_unc'].mean():.4f}")
print("="*80)


In [ ]:
# 13. Publication-Grade Visualizations (Figures 1-5)
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Figure 1: Risk-Coverage Pareto Curves
plt.figure(figsize=(7, 5), dpi=300)
plt.plot(cov_random * 100, r_random, '--', color='#7f8c8d', label=f'Random Referral (AURC={aurc_random:.3f})', linewidth=1.5)
plt.plot(cov_det * 100, r_det, '-', color='#e74c3c', label=f'Deterministic Entropy (AURC={aurc_det:.3f})', linewidth=2.0)
plt.plot(cov_mc * 100, r_mc, '-', color='#f39c12', label=f'MC Dropout Variance (AURC={aurc_mc:.3f})', linewidth=2.0)
plt.plot(cov_ens * 100, r_ens, '-', color='#2980b9', label=f'Deep Ensemble MutInfo (AURC={aurc_ens:.3f})', linewidth=2.2)
plt.xlabel('Coverage / Retention (%)', fontsize=12, fontweight='bold')
plt.ylabel('Empirical Risk (1.0 - Dice)', fontsize=12, fontweight='bold')
plt.title('Risk-Coverage Pareto Frontier for Selective Prediction', fontsize=13, fontweight='bold', pad=12)
plt.legend(frameon=True, fontsize=10, loc='upper left')
plt.xlim(20, 100)
plt.tight_layout()
plt.savefig('fig1_risk_coverage_curves.png', dpi=300)
plt.close()
print("Saved: fig1_risk_coverage_curves.png")

# Figure 2: Calibration Reliability Diagrams
plt.figure(figsize=(6.5, 5), dpi=300)
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=1.5)
plt.plot(confs_det, accs_det, 's-', color='#e74c3c', label=f'Deterministic (ESCE={esce_det:.3f})', markersize=6)
plt.plot(confs_mc, accs_mc, '^-', color='#f39c12', label=f'MC Dropout (ESCE={esce_mc:.3f})', markersize=6)
plt.plot(confs_ens, accs_ens, 'o-', color='#2980b9', label=f'Deep Ensemble (ESCE={esce_ens:.3f})', markersize=6)
plt.xlabel('Predicted Confidence', fontsize=12, fontweight='bold')
plt.ylabel('Empirical Pixel Accuracy', fontsize=12, fontweight='bold')
plt.title('Segmentation Reliability Diagrams (ESCE)', fontsize=13, fontweight='bold', pad=12)
plt.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.savefig('fig2_calibration_curves.png', dpi=300)
plt.close()
print("Saved: fig2_calibration_curves.png")

# Figure 3: Error Detection ROC
fpr_det, tpr_det, _ = roc_curve(targets_pool != (p_det_pool >= 0.5), det_entropies)
fpr_mc, tpr_mc, _ = roc_curve(targets_pool != (p_mc_pool >= 0.5), mc_variances)
fpr_ens, tpr_ens, _ = roc_curve(targets_pool != (p_ens_pool >= 0.5), ens_mis)

plt.figure(figsize=(6.5, 5), dpi=300)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.2)
plt.plot(fpr_det, tpr_det, color='#e74c3c', label=f'Deterministic (AUROC={df_results["det_auroc_ed"].mean():.3f})')
plt.plot(fpr_mc, tpr_mc, color='#f39c12', label=f'MC Dropout (AUROC={df_results["mc_auroc_ed"].mean():.3f})')
plt.plot(fpr_ens, tpr_ens, color='#2980b9', label=f'Deep Ensemble (AUROC={df_results["ens_auroc_ed"].mean():.3f})')
plt.xlabel('False Positive Rate (Correct Predicted Error)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Detected Error)', fontsize=12, fontweight='bold')
plt.title('AUROC for Segmentation Error Identification', fontsize=13, fontweight='bold', pad=12)
plt.legend(frameon=True, fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig('fig3_auroc_error_detection.png', dpi=300)
plt.close()
print("Saved: fig3_auroc_error_detection.png")

# Figure 4: AP vs PA Subgroups
sub_labels = ['AP View (Bedside)', 'PA View (Upright)']
det_sub = [df_results.loc[ap_mask & pos_mask, 'det_dice'].mean(), df_results.loc[pa_mask & pos_mask, 'det_dice'].mean()]
mc_sub = [df_results.loc[ap_mask & pos_mask, 'mc_dice'].mean(), df_results.loc[pa_mask & pos_mask, 'mc_dice'].mean()]
ens_sub = [df_results.loc[ap_mask & pos_mask, 'ens_dice'].mean(), df_results.loc[pa_mask & pos_mask, 'ens_dice'].mean()]

x = np.arange(len(sub_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=300)
ax.bar(x - width, det_sub, width, label='Deterministic', color='#e74c3c')
ax.bar(x, mc_sub, width, label='MC Dropout', color='#f39c12')
ax.bar(x + width, ens_sub, width, label='Deep Ensemble', color='#2980b9')
ax.set_ylabel('Positive Dice Score (DSC_pos)', fontsize=12, fontweight='bold')
ax.set_title('Subgroup Segmentation Performance Across Projection Angles', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(sub_labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.savefig('fig4_ap_vs_pa_subgroups.png', dpi=300)
plt.close()
print("Saved: fig4_ap_vs_pa_subgroups.png")


In [ ]:
# 14. Save Comprehensive Metrics JSON Summary

summary_dict = {
    "dataset": {
        "total_images": len(df_splits),
        "train_images": len(train_df),
        "val_images": len(val_df),
        "test_images": len(test_df),
        "positive_prevalence": float(df_splits["HasPneumothorax"].mean()),
        "leakage_overlap": 0
    },
    "deterministic_exp03": {
        "dsc_pos_mean": float(df_results.loc[pos_mask, "det_dice"].mean()),
        "dsc_pos_std": float(df_results.loc[pos_mask, "det_dice"].std()),
        "dsc_all_mean": float(df_results["det_dice"].mean()),
        "iou_pos_mean": float(df_results.loc[pos_mask, "det_iou"].mean()),
        "sensitivity": float(df_results.loc[pos_mask, "det_sens"].mean()),
        "specificity": float(df_results["det_spec"].mean()),
        "auroc_ed": float(df_results["det_auroc_ed"].mean()),
        "esce": float(esce_det),
        "brier": float(brier_det),
        "aurc": float(aurc_det)
    },
    "mc_dropout_exp04": {
        "dsc_pos_mean": float(df_results.loc[pos_mask, "mc_dice"].mean()),
        "dsc_pos_std": float(df_results.loc[pos_mask, "mc_dice"].std()),
        "dsc_all_mean": float(df_results["mc_dice"].mean()),
        "iou_pos_mean": float(df_results.loc[pos_mask, "mc_iou"].mean()),
        "sensitivity": float(df_results.loc[pos_mask, "mc_sens"].mean()),
        "specificity": float(df_results["mc_spec"].mean()),
        "auroc_ed": float(df_results["mc_auroc_ed"].mean()),
        "esce": float(esce_mc),
        "brier": float(brier_mc),
        "aurc": float(aurc_mc)
    },
    "deep_ensemble_exp05": {
        "dsc_pos_mean": float(df_results.loc[pos_mask, "ens_dice"].mean()),
        "dsc_pos_std": float(df_results.loc[pos_mask, "ens_dice"].std()),
        "dsc_all_mean": float(df_results["ens_dice"].mean()),
        "iou_pos_mean": float(df_results.loc[pos_mask, "ens_iou"].mean()),
        "sensitivity": float(df_results.loc[pos_mask, "ens_sens"].mean()),
        "specificity": float(df_results["ens_spec"].mean()),
        "auroc_ed": float(df_results["ens_auroc_ed"].mean()),
        "esce": float(esce_ens),
        "brier": float(brier_ens),
        "aurc": float(aurc_ens)
    },
    "hypothesis_tests": {
        "h1_ens_vs_det_dsc_pos_pvalue": float(p_h1),
        "h2_mc_vs_det_auroc_ed_pvalue": float(p_h2),
        "h3_ens_vs_det_auroc_ed_pvalue": float(p_h3)
    },
    "selective_prediction_triage": {
        "aurc_random": float(aurc_random),
        "aurc_det": float(aurc_det),
        "aurc_mc": float(aurc_mc),
        "aurc_ens": float(aurc_ens),
        "dice_at_100_cov": {"det": ret_det[0], "mc": ret_mc[0], "ens": ret_ens[0]},
        "dice_at_90_cov": {"det": ret_det[1], "mc": ret_mc[1], "ens": ret_ens[1]},
        "dice_at_80_cov": {"det": ret_det[2], "mc": ret_mc[2], "ens": ret_ens[2]},
        "dice_at_70_cov": {"det": ret_det[3], "mc": ret_mc[3], "ens": ret_ens[3]}
    }
}

with open("metrics_summary.json", "w") as f:
    json.dump(summary_dict, f, indent=2)

print("Comprehensive metrics summary saved to metrics_summary.json")
print("ALL EXPERIMENTS EXP-03 THROUGH EXP-10 COMPLETED SUCCESSFULLY.")
